# VITS Bengali TTS Model Training

This notebook demonstrates training a VITS (Variational Inference with adversarial learning for end-to-end Text-to-Speech) model on Bengali language audio data.

## 1. Install Dependencies

Install the TTS library, espeak-ng phonemizer, gdown for downloading files, and unzip utility.

In [ ]:
%%capture
!pip install TTS
!sudo apt-get install espeak-ng -y
!pip install gdown
!sudo apt-get install unzip

## 2. Download and Extract Dataset

Download the Bengali audio dataset from Google Drive and extract it to the working directory.

In [ ]:
!gdown "https://drive.google.com/uc?id=your_id"
!unzip /home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/data.zip -d /home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/data

## 3. Download Pretrained Model from Hugging Face

Use snapshot_download to download a pretrained VITS checkpoint from Hugging Face Hub for transfer learning.

In [ ]:
from huggingface_hub import snapshot_download

local_data_dir = "/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/model/male_vits_23_dec_2024-December-23-2024_09+14AM-0000000"
dataset_repo = "sifat1221/vits_bn_tts_checkpoint_125000"
file_path = snapshot_download(repo_id=dataset_repo, local_dir=local_data_dir)
print(f"Model downloaded to: {file_path}")

## 4. Verify Dataset Files

Count and verify the number of audio files in the dataset directory to ensure successful download.

In [ ]:
import os

folder_path = "/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/data/data/wav"
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in '{folder_path}': {file_count}")

## 5. Import Required Libraries

Import necessary libraries including NumPy, Pandas, PyTorch, TTS library components, and trainer utilities.

In [ ]:
import numpy as np
import pandas as pd
import os
import torch

from trainer import Trainer, TrainerArgs
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.utils.audio import AudioProcessor
from TTS.tts.datasets import load_tts_samples
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.tts.models.vits import Vits, VitsAudioConfig, CharactersConfig

## 6. Configure Environment Paths

Detect the execution environment (Colab, Kaggle, or Lightning) and set appropriate data and model paths.

In [ ]:
if "COLAB_GPU" in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    data_root = '/content/drive/MyDrive/IIT/4-2/ML/ML-Project_Team_minions/data'
    model_root = '/content/drive/MyDrive/IIT/4-2/ML/ML-Project_Team_minions/Models'
    print("Running in Google Colab")
    
elif os.path.exists("/kaggle"):
    print("Running in Kaggle")
    data_root = '/kaggle/input/bangla-audio/data'
    model_root = '/kaggle/working/'

else:
    print("Running in Local/Lightning environment")
    data_root = '/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/data/data'
    model_root = '/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/model'

print(f"Data root: {data_root}")
print(f"Model root: {model_root}")

## 7. Set Training Parameters

Configure gender-specific training (male/female), pretrained model usage, and metadata file paths.

In [ ]:
# Training configuration
male = True
pretrained = True
pretrained_path = ""

if pretrained:
    pretrained_path = '/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/model/male_vits_23_dec_2024-December-23-2024_09+14AM-0000000'
    print(f"Using pretrained model from: {pretrained_path}")
    
# Set metadata file and root path based on gender
if male:
    meta_file = f'{data_root}/metadata.csv'
    root_path = f'{data_root}'
    print("Training on MALE voice dataset")
else:
    meta_file = f'{data_root}/female/mono/metadata_female.txt'
    root_path = f'{data_root}/female/mono'
    print("Training on FEMALE voice dataset")

print(f"Metadata file: {meta_file}")
print(f"Root path: {root_path}")

## 8. Define Custom Data Formatter

Create a formatter function to parse the metadata file and prepare audio-text pairs for training.

In [ ]:
def formatter(root_path, meta_file, **kwargs):
    """
    Custom formatter to parse metadata file and create training samples.
    
    Args:
        root_path: Root directory containing audio files
        meta_file: Path to metadata file containing audio-text pairs
        
    Returns:
        List of dictionaries containing text, audio_file, speaker_name, and root_path
    """
    txt_file = meta_file
    items = []
    speaker_name = "ljspeech"
    
    with open(txt_file, "r", encoding="utf-8") as ttf:
        for line in ttf:
            cols = line.split("|")
            wav_file = os.path.join(root_path, "wav", cols[0] + ".wav")
            
            text = ''
            try:
                text = cols[2].strip()
            except:
                print(f"Warning: Could not extract text for {cols[0]}")
                continue
                
            items.append({
                "text": text, 
                "audio_file": wav_file, 
                "speaker_name": speaker_name, 
                "root_path": root_path
            })
    
    print(f"Loaded {len(items)} samples from metadata")
    return items

## 9. Load Training and Evaluation Samples

Use load_tts_samples with the custom formatter to split the dataset into training and evaluation sets with an 80-20 split.

In [ ]:
# Create dataset configuration
dataset_config = BaseDatasetConfig(
    meta_file_train=meta_file, 
    path=os.path.join(root_path, "")
)

# Load and split samples
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    formatter=formatter, 
    eval_split=True, 
    eval_split_size=0.2
)

print(f"Training samples: {len(train_samples)}")
print(f"Evaluation samples: {len(eval_samples)}")
print(f"Total samples: {len(train_samples) + len(eval_samples)}")

## 10. Configure VITS Model and Audio Settings

Set up VitsConfig with audio parameters, batch sizes, phonemizer settings for Bengali language, and initialize the model with tokenizer.

In [ ]:
# Set output path and phoneme cache
output_path = model_root
phoneme_cache_path = os.path.join(output_path, "phoneme_cache")
os.makedirs(output_path, exist_ok=True)
os.makedirs(phoneme_cache_path, exist_ok=True)

# Configure audio processing parameters
audio_config = VitsAudioConfig(
    sample_rate=22050, 
    win_length=1024, 
    hop_length=256, 
    num_mels=80, 
    mel_fmin=0, 
    mel_fmax=None
)

# Configure VITS model
config = VitsConfig(
    audio=audio_config,
    run_name="male_vits_23_dec_2024",
    batch_size=48,
    eval_batch_size=32,
    epochs=1000,
    save_step=5000,
    print_step=500,
    batch_group_size=0,
    num_loader_workers=8,
    num_eval_loader_workers=8,
    run_eval=True,
    test_delay_epochs=-1,
    phonemizer="espeak",
    text_cleaner='phoneme_cleaners',
    use_phonemes=True,
    phoneme_language="bn",  # Bengali language code
    phoneme_cache_path=phoneme_cache_path,
    compute_input_seq_cache=True,
    add_blank=True,
    use_language_weighted_sampler=True,
    print_eval=False,
    mixed_precision=False,
    output_path=output_path,
    datasets=[dataset_config],
    cudnn_benchmark=True,
    test_sentences=[
        'আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।',
        'চিরদিন তোমার আকাশ, তোমার বাতাস, আমার প্রাণে বাজায় বাঁশি',
        'ও মা, ফাগুনে তোর আমের বনে ঘ্রাণে পাগল করে,মরি হায়, হায় রে।'
    ]
)

print("Configuration created successfully")
print(f"Output path: {output_path}")
print(f"Phoneme cache path: {phoneme_cache_path}")

## 11. Initialize Model and Trainer

Create VITS model instance with the configuration and initialize the Trainer with training arguments, optionally loading from pretrained checkpoint.

In [ ]:
# Initialize audio processor
ap = AudioProcessor.init_from_config(config)
print("Audio processor initialized")

# Initialize tokenizer
tokenizer, config = TTSTokenizer.init_from_config(config)
print("Tokenizer initialized")

# Initialize VITS model
model = Vits(config, ap, tokenizer, speaker_manager=None)
print("VITS model initialized")

# Initialize trainer with optional pretrained checkpoint
trainer_args = TrainerArgs(continue_path=pretrained_path if pretrained else None)
trainer = Trainer(
    trainer_args, 
    config, 
    output_path, 
    model=model, 
    train_samples=train_samples, 
    eval_samples=eval_samples
)
print("Trainer initialized")

if pretrained:
    print(f"Continuing training from: {pretrained_path}")
else:
    print("Starting training from scratch")

## 12. Train the Model

Execute the training loop using trainer.fit() and monitor the training progress.

In [ ]:
%%time
print("Starting training...")
trainer.fit()
print("Training completed!")

## 13. Upload Model to Hugging Face

Upload the trained model checkpoint to Hugging Face Hub for sharing and deployment.

In [ ]:
from huggingface_hub import HfApi, upload_folder

# Configuration for Hugging Face upload
api_token = "your_api_token"  # Replace with your Hugging Face token
repo_id = "sifat1221/vits_bn_tts_checkpoint_140000"  # Replace with your repo ID
local_dir = '/home/tr/MEGA/programming2/L4T2/CSE472_ML_Project/model/male_vits_23_dec_2024-December-23-2024_09+14AM-0000000'

# Initialize Hugging Face API
api = HfApi(token=api_token)

# Create repository (if it doesn't exist)
api.create_repo(
    repo_id=repo_id, 
    repo_type="model", 
    private=False, 
    exist_ok=True
)
print(f"Repository '{repo_id}' created/verified")

# Upload model files
upload_folder(
    folder_path=local_dir,
    path_in_repo="",
    repo_id=repo_id,
    token=api_token,
    repo_type="model",
)
print(f"Files from '{local_dir}' uploaded to the repository '{repo_id}' successfully.")

## Training Complete

The VITS model has been trained on Bengali audio data and uploaded to Hugging Face Hub. You can now use this model for Bengali text-to-speech synthesis.